***Load the Dataset***

In [3]:
import pandas as pd

df_tweet = pd.read_csv('/content/Tweets.csv')
df_tweet=df_tweet[["airline_sentiment", "text"]]

df_tweet.head()

,airline_sentiment,text
0,neutral,@VirginAmerica What @dhepburn said.
1,positive,@VirginAmerica plus you've added commercials t...
2,neutral,@VirginAmerica I didn't today... Must mean I n...
3,negative,@VirginAmerica it's really aggressive to blast...
4,negative,@VirginAmerica and it's a really big bad thing...


Text Preprocessing

In [11]:
import nltk
import string
import re

from nltk.stem.porter import PorterStemmer

nltk.download('stopwords')
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('punkt_tab') # Added to download the missing resource

ps = PorterStemmer()

def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\\S+', '', text)

    # Tokenization
    text = nltk.word_tokenize(text)

    # Remove stopwords and punctuation
    y = []
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)

     # Stemming
    stemmed = []
    for i in y:
        stemmed.append(ps.stem(i))

    return " ".join(stemmed)

df_tweet["text_cleaned"] = df_tweet["text"].apply(clean_text)

# Check output
print(df_tweet[["text", "text_cleaned"]].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


                                                text  \
0                @VirginAmerica What @dhepburn said.   
1  @VirginAmerica plus you've added commercials t...   
2  @VirginAmerica I didn't today... Must mean I n...   
3  @VirginAmerica it's really aggressive to blast...   
4  @VirginAmerica and it's a really big bad thing...   

                                        text_cleaned  
0                        virginamerica dhepburn said  
1  virginamerica plu 've ad commerci experi ... t...  
2  virginamerica n't today ... must mean need tak...  
3  virginamerica 's realli aggress blast obnoxi `...  
4              virginamerica 's realli big bad thing  


 Feature Extraction

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create vectorizer
tfidf = TfidfVectorizer(max_features=3000)

# Transform text into vectors
X = tfidf.fit_transform(df_tweet["text_cleaned"]).toarray()

# Labels
Y = df_tweet["airline_sentiment"].values

Train Models

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2
)

In [15]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.7219945355191257


In [16]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred))

Random Forest Accuracy: 0.75
